# Task 1 - Iris Flower Classification
**Internship:** CodeAlpha (Data Science / Machine Learning Track)

**Objective:** Use measurements of Iris flowers (setosa, versicolor, virginica) to train a machine learning model that classifies the species based on those measurements. Evaluate model accuracy and understand basic classification concepts.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Libraries loaded')

## Step 2: Load the Iris Dataset

The Iris dataset is bundled directly with Scikit-learn, so no manual download is needed -- this matches the task's suggestion to use Scikit-learn for easy dataset access.

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].map({0:'setosa', 1:'versicolor', 2:'virginica'})

df.to_csv('iris_dataset.csv', index=False)
print('Shape:', df.shape)
df.head()

## Step 3: Explore the Dataset

In [ ]:
print('Species distribution:')
print(df['species_name'].value_counts())
print()
print('Summary statistics:')
df.describe()

## Step 4: Visualize Sepal Measurements by Species

In [ ]:
colors = {'setosa':'#E74C3C','versicolor':'#3498DB','virginica':'#2ECC71'}

plt.figure(figsize=(7,5))
for sp in df['species_name'].unique():
    subset = df[df['species_name']==sp]
    plt.scatter(subset['sepal length (cm)'], subset['sepal width (cm)'], label=sp, color=colors[sp], alpha=0.7)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.title('Sepal Length vs Width by Species')
plt.legend()
plt.tight_layout()
plt.savefig('sepal_scatter.png')
plt.show()

## Step 5: Visualize Petal Measurements by Species

Petal measurements typically separate the species far more cleanly than sepal measurements.

In [ ]:
plt.figure(figsize=(7,5))
for sp in df['species_name'].unique():
    subset = df[df['species_name']==sp]
    plt.scatter(subset['petal length (cm)'], subset['petal width (cm)'], label=sp, color=colors[sp], alpha=0.7)
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('Petal Length vs Width by Species')
plt.legend()
plt.tight_layout()
plt.savefig('petal_scatter.png')
plt.show()

## Step 6: Feature Distributions by Species

In [ ]:
features = ['sepal length (cm)','sepal width (cm)','petal length (cm)','petal width (cm)']

fig, axes = plt.subplots(2, 2, figsize=(10,8))
for ax, feat in zip(axes.flatten(), features):
    sns.boxplot(x='species_name', y=feat, data=df, hue='species_name', palette=colors, legend=False, ax=ax)
    ax.set_title(feat)
plt.tight_layout()
plt.savefig('feature_boxplots.png')
plt.show()

## Step 7: Feature Correlation

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.show()

## Step 8: Train/Test Split

In [ ]:
X = df[features]
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

## Step 9: Train Multiple Classification Models

Comparing a few common classification algorithms to see how each performs on this dataset.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='linear', random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f'{name}: {acc:.4f}')

In [ ]:
plt.figure(figsize=(7,4))
plt.bar(results.keys(), results.values(), color=['#9B59B6','#16A085','#E67E22'])
plt.ylabel('Accuracy')
plt.title('Model Comparison - Test Accuracy')
plt.ylim(0,1.1)
for i, (k,v) in enumerate(results.items()):
    plt.text(i, v+0.02, f'{v:.2%}', ha='center')
plt.tight_layout()
plt.savefig('model_comparison.png')
plt.show()

## Step 10: Detailed Evaluation of Best Model

In [ ]:
best_model = models['Random Forest']
best_preds = best_model.predict(X_test)

print('Classification Report (Random Forest):')
print(classification_report(y_test, best_preds, target_names=['setosa','versicolor','virginica']))

In [ ]:
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['setosa','versicolor','virginica'],
            yticklabels=['setosa','versicolor','virginica'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Random Forest')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

## Step 11: Feature Importance

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=features).sort_values()

plt.figure(figsize=(7,4))
importances.plot(kind='barh', color='#16A085')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

print(importances)

## Key Concepts Demonstrated

- **Classification**: predicting a category (species) rather than a continuous number
- **Train/test split**: holding out data the model never sees during training, to fairly judge performance
- **Multiple models comparison**: Decision Tree, Random Forest, and SVM all solve the same problem differently
- **Confusion matrix**: shows exactly which species get confused with which
- **Feature importance**: petal length and petal width are far more useful for distinguishing species than sepal measurements

## Summary

| Model | Accuracy |
|---|---|
| Decision Tree | ~93% |
| Random Forest | ~90% |
| SVM (linear) | ~100% |

**Conclusion:** Petal length and petal width are the strongest predictors of Iris species -- setosa is almost perfectly separable, while versicolor and virginica have some overlap, which is reflected in the confusion matrix and classification report.